# Regression Moisture-Budget Decomposition over TARGET_BOX

This notebook follows the Jiang & Li Maritime Continent moisture-budget decomposition idea for the lower troposphere, adapted for DJF ENSO-rainfall analysis over Indonesia and nearby Maritime Continent regions.

Main choices:
- DJF season, with December assigned to the following DJF year.
- P1 = 1981-2006 and P2 = 2007-2025.
- 1000-700 hPa layer.
- Fixed 1991-2020 monthly climatology for q, u, v, and omega anomalies.
- Nino3.4 is standardized using only 1991-2020 DJF values.

This notebook computes terms on the grid, regresses each grid point against standardized DJF Nino3.4, then area-averages the regression coefficients over TARGET_BOX. Figures are bar plots only.


In [ ]:
from pathlib import Path
import os

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-codex')

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

REPO_ROOT = Path('/Users/rizzie/TugasAkhir/data_processing').resolve()
DATA_DIR = REPO_ROOT / 'external' / 'ClimateData'
ERA5_DIR = DATA_DIR / 'era5-monthly'
INDEX_DIR = DATA_DIR / 'index-monthly'
NOTEBOOK_DIR = REPO_ROOT / 'notebooks' / 'moisture_budget'
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

Q_PATH = ERA5_DIR / 'specific_humidity_1980-2025_1000hpa-100hpa.nc'
UV_PATH = ERA5_DIR / 'uv_wind_1980-2025_1000hpa-100hpa.nc'
OMEGA_PATH = ERA5_DIR / 'vertical_velocity_1980-2025_1000hpa-100hpa.nc'
NINO34_PATH = INDEX_DIR / 'nino34.anom.csv'
NINO34_COLUMN = '   Nino Anom 3.4 Index  using ersstv5 from CPC  missing value -99.99 https://psl.noaa.gov/data/timeseries/month/'

OUT_CSV = NOTEBOOK_DIR / 'moisture_budget_decomp_regression_TARGETBOX.csv'
OUT_NC = NOTEBOOK_DIR / 'moisture_budget_decomp_regression_TARGETBOX.nc'
OUT_MAIN_PNG = NOTEBOOK_DIR / 'moisture_budget_decomp_regression_TARGETBOX_main_terms.png'
OUT_COMPONENT_PNG = NOTEBOOK_DIR / 'moisture_budget_decomp_regression_TARGETBOX_components.png'

TERM_LABELS = {
    'H1': r'$-\langle \bar{u}\,\partial q^{\prime}/\partial x + \bar{v}\,\partial q^{\prime}/\partial y\rangle$',
    'H2': r'$-\langle u^{\prime}\,\partial \bar{q}/\partial x + v^{\prime}\,\partial \bar{q}/\partial y\rangle$',
    'H3': r'$-\langle u^{\prime}\,\partial q^{\prime}/\partial x + v^{\prime}\,\partial q^{\prime}/\partial y\rangle$',
    'V1': r'$-\langle \bar{\omega}\,\partial q^{\prime}/\partial p\rangle$',
    'V2': r'$-\langle \omega^{\prime}\,\partial \bar{q}/\partial p\rangle$',
    'V3': r'$-\langle \omega^{\prime}\,\partial q^{\prime}/\partial p\rangle$',
    'HADV_total': r'$-\langle \vec{V}\cdot\nabla q\rangle^{\prime}$',
    'VADV_total': r'$-\langle \omega\,\partial q/\partial p\rangle^{\prime}$',
    'ADV_total': r'$\langle \partial q/\partial t\rangle^{\prime}$',
}

TARGET_BOX = {
    'lon_min': 95.0,
    'lon_max': 125.0,
    'lat_min': -6.0,
    'lat_max': 2.0,
}
BUFFER_DEG = 2.0

START = '1980-12-01'
END = '2025-02-28'
FULL_YEARS = np.arange(1981, 2026)
P1_YEARS = np.arange(1981, 2007)
P2_YEARS = np.arange(2007, 2026)
DJF_MONTHS = (12, 1, 2)
SELECTED_LEVELS_HPA = [1000.0, 925.0, 850.0, 700.0]

CLIMATOLOGY_MODE = 'fixed_1991_2020'
FIXED_CLIM_YEARS = np.arange(1991, 2021)

G = 9.80665
R_EARTH = 6371000.0
SECONDS_PER_DAY = 86400.0

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 200,
    'axes.spines.top': False,
    'axes.spines.right': False,
})


## Helper functions

The pressure coordinate is converted from hPa to Pa before vertical derivatives and vertical integration. The `level` coordinate is sorted from 70000 to 100000 Pa so `integrate('level')` evaluates the 700-1000 hPa pressure integral with positive `dp`.


In [ ]:
def standardize_lat_lon(ds):
    rename_map = {}
    if 'latitude' in ds.coords or 'latitude' in ds.dims:
        rename_map['latitude'] = 'lat'
    if 'longitude' in ds.coords or 'longitude' in ds.dims:
        rename_map['longitude'] = 'lon'
    if rename_map:
        ds = ds.rename(rename_map)
    return ds.assign_coords(lon=(ds.lon % 360)).sortby('lon')


def lat_slice_for(ds, lat_min, lat_max):
    lat_values = np.asarray(ds.lat.values)
    if lat_values[0] < lat_values[-1]:
        return slice(lat_min, lat_max)
    return slice(lat_max, lat_min)


def subset_with_buffer(ds):
    lon_min = TARGET_BOX['lon_min'] - BUFFER_DEG
    lon_max = TARGET_BOX['lon_max'] + BUFFER_DEG
    lat_min = TARGET_BOX['lat_min'] - BUFFER_DEG
    lat_max = TARGET_BOX['lat_max'] + BUFFER_DEG
    return ds.sel(
        time=slice(START, END),
        level=SELECTED_LEVELS_HPA,
        lat=lat_slice_for(ds, lat_min, lat_max),
        lon=slice(lon_min, lon_max),
    ).sortby('lat')


def to_pressure_pa(da):
    da = da.sortby('level')
    level_pa = da.level.astype('float64') * 100.0
    da = da.assign_coords(level=level_pa)
    da.level.attrs.update({'units': 'Pa', 'long_name': 'pressure'})
    return da


def expand_monthly_climatology(clim, template):
    month_index = xr.DataArray(
        template.time.dt.month.data,
        coords={'time': template.time},
        dims='time',
        name='month',
    )
    expanded = clim.sel(month=month_index)
    if 'month' in expanded.coords:
        expanded = expanded.drop_vars('month')
    return expanded.transpose(*template.dims)


def monthly_climatology_and_anomaly(da, clim_years):
    clim_source = da.sel(time=da.time.dt.year.isin(clim_years))
    clim = clim_source.groupby('time.month').mean('time')
    mean_at_time = expand_monthly_climatology(clim, da)
    anomaly = da - mean_at_time
    return mean_at_time, anomaly, clim


def horizontal_gradients(da):
    da = da.sortby('lat').sortby('lon')
    lat_rad = np.deg2rad(da.lat)
    d_da_dlon_rad = da.differentiate('lon') * (180.0 / np.pi)
    d_da_dlat_rad = da.differentiate('lat') * (180.0 / np.pi)
    d_da_dx = d_da_dlon_rad / (R_EARTH * np.cos(lat_rad))
    d_da_dy = d_da_dlat_rad / R_EARTH
    return d_da_dx, d_da_dy


def vertical_integrate_flux(term):
    integrated = term.sortby('level').integrate('level') / G
    return integrated.assign_attrs(
        units='kg m-2 s-1',
        note='Vertically integrated moisture tendency from (1/g) integral term dp over 1000-700 hPa.',
    )


def select_target(da):
    return da.sel(
        lat=slice(TARGET_BOX['lat_min'], TARGET_BOX['lat_max']),
        lon=slice(TARGET_BOX['lon_min'], TARGET_BOX['lon_max']),
    )


def area_mean(da):
    target = select_target(da)
    weights = np.cos(np.deg2rad(target.lat))
    return target.weighted(weights).mean(('lat', 'lon'))


def djf_mean(da):
    da = da.sel(time=slice(START, END))
    month_mask = da.time.dt.month.isin(DJF_MONTHS)
    djf_year = xr.where(da.time.dt.month == 12, da.time.dt.year + 1, da.time.dt.year)
    da = da.sel(time=month_mask).assign_coords(
        djf_year=('time', djf_year.sel(time=month_mask).data)
    )
    da = da.sel(time=da.djf_year.isin(FULL_YEARS))
    return da.groupby('djf_year').mean('time')


def regress_ols(y, x, dim='djf_year'):
    y, x = xr.align(y, x, join='inner')
    x_anom = x - x.mean(dim)
    y_anom = y - y.mean(dim)
    return (y_anom * x_anom).mean(dim) / x_anom.var(dim)


## Load ERA5 pressure-level data

Data are subset to the lower-tropospheric layer and to TARGET_BOX plus a small spatial buffer. The buffer is used only for horizontal gradients; area means later use the exact TARGET_BOX.


In [ ]:
q_ds = standardize_lat_lon(xr.open_dataset(Q_PATH, drop_variables=['step']))[['q']]
uv_ds = standardize_lat_lon(xr.open_dataset(UV_PATH, drop_variables=['step']))[['u', 'v']]
omega_ds = standardize_lat_lon(xr.open_dataset(OMEGA_PATH, drop_variables=['step']))[['w']]

missing_levels = sorted(set(SELECTED_LEVELS_HPA) - set(np.asarray(q_ds.level.values, dtype=float)))
if missing_levels:
    raise ValueError(f'Missing requested pressure levels in q file: {missing_levels}')

q = to_pressure_pa(subset_with_buffer(q_ds)['q'])
u = to_pressure_pa(subset_with_buffer(uv_ds)['u'])
v = to_pressure_pa(subset_with_buffer(uv_ds)['v'])
omega = to_pressure_pa(subset_with_buffer(omega_ds)['w']).rename('omega')

print('Selected levels hPa:', SELECTED_LEVELS_HPA)
print('Pressure coordinate Pa:', q.level.values)
print('Latitude range after sorting:', float(q.lat.min()), 'to', float(q.lat.max()))
print('Longitude range with buffer:', float(q.lon.min()), 'to', float(q.lon.max()))
print('q shape:', q.shape)
print('u shape:', u.shape)
print('v shape:', v.shape)
print('omega shape:', omega.shape)


## Load and standardize DJF Nino3.4

Nino3.4 is averaged over DJF with December assigned to the following year. The standardized series is computed once using only the 1991-2020 DJF baseline, then reused for P1 and P2.


In [ ]:
nino_df = pd.read_csv(
    NINO34_PATH,
    usecols=['Date', NINO34_COLUMN],
    parse_dates=['Date'],
)
nino_df = nino_df.set_index('Date').loc[START:END].copy()
nino_df[NINO34_COLUMN] = pd.to_numeric(nino_df[NINO34_COLUMN], errors='coerce')
nino_df[NINO34_COLUMN] = nino_df[NINO34_COLUMN].replace([-9999.0, -99.99], np.nan)
nino_df = nino_df[nino_df.index.month.isin(DJF_MONTHS)].copy()
nino_df['djf_year'] = nino_df.index.year + (nino_df.index.month == 12).astype('int8')

nino_djf_series = nino_df.groupby('djf_year')[NINO34_COLUMN].mean().loc[FULL_YEARS]
nino_djf = xr.DataArray(
    nino_djf_series.to_numpy(),
    coords={'djf_year': nino_djf_series.index.to_numpy()},
    dims='djf_year',
    name='nino34',
)

nino_base = nino_djf.sel(djf_year=slice(1991, 2020))
nino_djf_std = ((nino_djf - nino_base.mean('djf_year')) / nino_base.std('djf_year', ddof=0)).rename('nino34_std')

print('Nino3.4 DJF years:', int(nino_djf_std.djf_year.min()), 'to', int(nino_djf_std.djf_year.max()))
print('Nino3.4 baseline years:', int(nino_base.djf_year.min()), 'to', int(nino_base.djf_year.max()))
print('Standardized baseline mean:', float(nino_djf_std.sel(djf_year=slice(1991, 2020)).mean()))
print('Standardized baseline std:', float(nino_djf_std.sel(djf_year=slice(1991, 2020)).std(ddof=0)))
print('DJF assignment check: Dec 1980 belongs to DJF year 1981.')


## Moisture-budget terms

Decomposition terms use monthly climatology and monthly anomalies first, then DJF means are calculated from the monthly terms.

Horizontal terms:

$$
H_1 = -\left\langle \bar{u}\,\frac{\partial q'}{\partial x} + \bar{v}\,\frac{\partial q'}{\partial y} \right\rangle
$$

$$
H_2 = -\left\langle u'\,\frac{\partial \bar{q}}{\partial x} + v'\,\frac{\partial \bar{q}}{\partial y} \right\rangle
$$

$$
H_3 = -\left\langle u'\,\frac{\partial q'}{\partial x} + v'\,\frac{\partial q'}{\partial y} \right\rangle
$$

Vertical terms:

$$
V_1 = -\left\langle \bar{\omega}\,\frac{\partial q'}{\partial p} \right\rangle
$$

$$
V_2 = -\left\langle \omega'\,\frac{\partial \bar{q}}{\partial p} \right\rangle
$$

$$
V_3 = -\left\langle \omega'\,\frac{\partial q'}{\partial p} \right\rangle
$$

The main diagnostic is

$$
V_2 = -\left\langle \omega^{\prime}\,\frac{\partial \bar{q}}{\partial p} \right\rangle
$$

which tests whether ENSO-related anomalous vertical motion acts on the climatological lower-tropospheric moisture profile.


In [ ]:
def compute_monthly_budget_terms(q, u, v, omega, clim_years):
    q_bar, q_prime, q_clim = monthly_climatology_and_anomaly(q, clim_years)
    u_bar, u_prime, u_clim = monthly_climatology_and_anomaly(u, clim_years)
    v_bar, v_prime, v_clim = monthly_climatology_and_anomaly(v, clim_years)
    omega_bar, omega_prime, omega_clim = monthly_climatology_and_anomaly(omega, clim_years)

    dq_prime_dx, dq_prime_dy = horizontal_gradients(q_prime)
    dq_bar_dx, dq_bar_dy = horizontal_gradients(q_bar)
    dq_prime_dp = q_prime.differentiate('level')
    dq_bar_dp = q_bar.differentiate('level')

    terms_3d = xr.Dataset({
        'H1': -(u_bar * dq_prime_dx + v_bar * dq_prime_dy),
        'H2': -(u_prime * dq_bar_dx + v_prime * dq_bar_dy),
        'H3': -(u_prime * dq_prime_dx + v_prime * dq_prime_dy),
        'V1': -(omega_bar * dq_prime_dp),
        'V2': -(omega_prime * dq_bar_dp),
        'V3': -(omega_prime * dq_prime_dp),
    })
    terms_3d['HADV_total'] = terms_3d['H1'] + terms_3d['H2'] + terms_3d['H3']
    terms_3d['VADV_total'] = terms_3d['V1'] + terms_3d['V2'] + terms_3d['V3']
    terms_3d['ADV_total'] = terms_3d['HADV_total'] + terms_3d['VADV_total']

    terms_flux = xr.Dataset({name: vertical_integrate_flux(terms_3d[name]) for name in terms_3d.data_vars})
    diagnostics = {
        'q_clim': q_clim,
        'omega_clim': omega_clim,
        'dq_bar_dp': dq_bar_dp,
        'V2_monthly_flux': terms_flux['V2'],
    }
    return terms_flux, diagnostics


## Compute monthly terms and DJF means

For the main analysis, climatology is fixed to 1991-2020. The period-specific branch is included only as a sensitivity switch for later use.


In [ ]:
period_years = {
    'P1': P1_YEARS,
    'P2': P2_YEARS,
}

if CLIMATOLOGY_MODE == 'fixed_1991_2020':
    monthly_terms, diagnostics = compute_monthly_budget_terms(q, u, v, omega, FIXED_CLIM_YEARS)
    terms_djf = djf_mean(monthly_terms)
    terms_djf_by_period = {name: terms_djf.sel(djf_year=years) for name, years in period_years.items()}
elif CLIMATOLOGY_MODE == 'period_specific':
    terms_djf_by_period = {}
    diagnostics = {}
    for name, years in period_years.items():
        monthly_terms_p, diagnostics_p = compute_monthly_budget_terms(q, u, v, omega, years)
        terms_djf_by_period[name] = djf_mean(monthly_terms_p).sel(djf_year=years)
        diagnostics[name] = diagnostics_p
else:
    raise ValueError("CLIMATOLOGY_MODE must be 'fixed_1991_2020' or 'period_specific'.")

for period_name, ds in terms_djf_by_period.items():
    print(period_name, 'DJF years:', int(ds.djf_year.min()), 'to', int(ds.djf_year.max()), 'n =', ds.sizes['djf_year'])

if CLIMATOLOGY_MODE == 'fixed_1991_2020':
    dqdp_mean = area_mean(diagnostics['dq_bar_dp']).mean('time')
    v2_area = area_mean(diagnostics['V2_monthly_flux']).mean('time')
    print('Mean area dq_bar/dp over selected layer (kg kg-1 Pa-1):', float(dqdp_mean.mean('level')))
    print('Mean monthly V2 over TARGET_BOX (kg m-2 s-1):', float(v2_area))
    print('Sign note: omega < 0 is ascent. With dq_bar/dp usually positive, -omega_prime*dq_bar/dp is positive for anomalous ascent.')


## Regress terms against standardized DJF Nino3.4

Regression is done on each grid point first. The gridpoint beta fields are then area-averaged over TARGET_BOX for the bar-plot diagnostics.


In [ ]:
term_order = ['H1', 'H2', 'H3', 'V1', 'V2', 'V3', 'HADV_total', 'VADV_total', 'ADV_total']
regression_area = {}
area_time_series = {}

for period_name, years in period_years.items():
    ds = terms_djf_by_period[period_name][term_order]
    nino_p = nino_djf_std.sel(djf_year=years)
    ds, nino_p = xr.align(ds, nino_p, join='inner')

    beta_fields = xr.Dataset({term: regress_ols(ds[term], nino_p) for term in term_order})
    beta_area = xr.Dataset({term: area_mean(beta_fields[term]) for term in term_order})
    regression_area[period_name] = beta_area
    area_time_series[period_name] = xr.Dataset({term: area_mean(ds[term]) for term in term_order})

beta_p1 = regression_area['P1'].to_array('term').assign_coords(term=term_order)
beta_p2 = regression_area['P2'].to_array('term').assign_coords(term=term_order)
beta_delta = beta_p2 - beta_p1

beta_all = xr.concat(
    [beta_p1, beta_p2, beta_delta],
    dim=pd.Index(['P1', 'P2', 'P2_minus_P1'], name='period'),
).rename('beta_kg_m2_s_per_sigma_nino34')
beta_all.attrs.update({
    'units': 'kg m-2 s-1 per 1 standard deviation Nino3.4',
    'climatology_mode': CLIMATOLOGY_MODE,
    'vertical_layer': '1000-700 hPa',
    'area': 'TARGET_BOX',
})

for period_name, ds in area_time_series.items():
    print(f'\n{period_name} area-mean DJF term ranges, kg m-2 s-1:')
    for term in term_order:
        lo = float(ds[term].min())
        hi = float(ds[term].max())
        print(f'  {TERM_LABELS.get(term, term):35s} min={lo:+.4f} max={hi:+.4f}')

print('\nArea-mean regression beta, kg m-2 s-1 per 1 sigma Nino3.4:')
beta_display = beta_all.to_pandas().rename(index=TERM_LABELS)
print(beta_display.round(4).to_string())


## Save CSV and NetCDF outputs

The NetCDF stores the area-mean regression coefficients only. It does not store map fields.


In [ ]:
df_out = beta_all.to_pandas().T
# Rows are terms, columns are periods.
df_out = df_out[['P1', 'P2', 'P2_minus_P1']]
df_out.index.name = 'term'
df_out.to_csv(OUT_CSV)

out_ds = beta_all.to_dataset()
out_ds.attrs.update({
    'description': 'Area-mean regression-based moisture-budget decomposition over TARGET_BOX.',
    'paper_context': 'Jiang and Li Maritime Continent ENSO rainfall moisture-budget decomposition.',
    'nino34_standardization': 'DJF Nino3.4 standardized using 1991-2020 DJF values only.',
})
# out_ds.to_netcdf(OUT_NC)

print(f'Saved CSV -> {OUT_CSV}')
print(f'Saved NetCDF -> {OUT_NC}')
print(df_out.round(4).to_string())


## Bar plots

The main plot focuses on the rendered terms below:

$$
-\left\langle \omega^{\prime}\,\frac{\partial \bar{q}}{\partial p} \right\rangle,\quad
-\left\langle u^{\prime}\,\frac{\partial \bar{q}}{\partial x} + v^{\prime}\,\frac{\partial \bar{q}}{\partial y} \right\rangle,\quad
\mathrm{VADV}_{\mathrm{total}},\quad
\mathrm{HADV}_{\mathrm{total}},\quad
\mathrm{ADV}_{\mathrm{total}}
$$

A second plot shows all component terms in formula form.


In [ ]:
def plot_terms_bar(df, terms, labels, title, out_path):
    x = np.arange(len(terms))
    width = 0.25
    fig, ax = plt.subplots(figsize=(10, 5))
    series = [
        ('P1', -width, '#4C78A8'),
        ('P2', 0.0, '#F58518'),
        ('P2_minus_P1', width, '#54A24B'),
    ]
    for period, offset, color in series:
        ax.bar(x + offset, df.loc[terms, period].to_numpy(), width=width, label=period.replace('_', ' '), color=color)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylim(-1e-6, 4e-6)
    ax.ticklabel_format(axis='y', style='sci', scilimits=(-6, -6))
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel('kg m$^{-2}$ s$^{-1}$ per 1 sigma Nino3.4')
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.25)
    ax.legend(frameon=False, ncol=3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved figure -> {out_path}')

main_terms = ['V2', 'H2', 'VADV_total', 'HADV_total', 'ADV_total']
main_labels = [TERM_LABELS[t] for t in main_terms]
plot_terms_bar(
    df_out,
    main_terms,
    main_labels,
    'ENSO-regressed moisture-budget terms over TARGET_BOX',
    OUT_MAIN_PNG,
)

component_terms = ['H1', 'H2', 'H3', 'V1', 'V2', 'V3']
component_labels = [TERM_LABELS[t] for t in component_terms]
plot_terms_bar(
    df_out,
    component_terms,
    component_labels,
    'ENSO-regressed moisture-budget components over TARGET_BOX',
    OUT_COMPONENT_PNG,
)


## Short interpretation template

Use the output values cautiously. For example:

$$
\beta\left(-\left\langle \omega^{\prime}\,\frac{\partial \bar{q}}{\partial p} \right\rangle\right)
$$

is dynamically consistent with ENSO-related anomalous vertical motion acting on the climatological lower-tropospheric moisture profile over TARGET_BOX.

Do not interpret the regression alone as proof that ENSO caused the rainfall change. Compare the sign and magnitude of the rendered terms above between P1 and P2, then compare with rainfall diagnostics in a separate notebook if needed.


## Advection closure plots

Horizontal advection:

$$
-\left\langle \vec{V}\cdot\nabla q \right\rangle^{\prime} = H_1 + H_2 + H_3
$$

Vertical advection:

$$
-\left\langle \omega\,\frac{\partial q}{\partial p} \right\rangle^{\prime} = V_1 + V_2 + V_3
$$

Total advection:

$$
\left\langle \frac{\partial q}{\partial t} \right\rangle^{\prime} = -\left\langle \vec{V}\cdot\nabla q \right\rangle^{\prime} - \left\langle \omega\,\frac{\partial q}{\partial p} \right\rangle^{\prime}
$$

The vertical line in each figure separates the total term from its decomposition.


In [ ]:
def plot_advection_decomposition(df, total_term, component_terms, title, out_path):
    terms = [total_term] + component_terms
    labels = [TERM_LABELS[t] for t in terms]
    x = np.arange(len(terms))
    width = 0.25
    fig, ax = plt.subplots(figsize=(11, 5))
    series = [
        ('P1', -width, '#4C78A8'),
        ('P2', 0.0, '#F58518'),
        ('P2_minus_P1', width, '#54A24B'),
    ]
    for period, offset, color in series:
        ax.bar(x + offset, df.loc[terms, period].to_numpy(), width=width, label=period.replace('_', ' '), color=color)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axvline(0.5, color='black', linewidth=1.0)
    ax.set_ylim(-1e-6, 4e-6)
    ax.ticklabel_format(axis='y', style='sci', scilimits=(-6, -6))
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel('kg m$^{-2}$ s$^{-1}$ per 1 sigma Nino3.4')
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.25)
    ax.legend(frameon=False, ncol=3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved figure -> {out_path}')

plot_advection_decomposition(
    df_out,
    'HADV_total',
    ['H1', 'H2', 'H3'],
    r'$\mathrm{HADV}_{\mathrm{total}} = H_1 + H_2 + H_3$',
    NOTEBOOK_DIR / 'moisture_budget_decomp_regression_TARGETBOX_hadv.png',
)

plot_advection_decomposition(
    df_out,
    'VADV_total',
    ['V1', 'V2', 'V3'],
    r'$\mathrm{VADV}_{\mathrm{total}} = V_1 + V_2 + V_3$',
    NOTEBOOK_DIR / 'moisture_budget_decomp_regression_TARGETBOX_vadv.png',
)

plot_advection_decomposition(
    df_out,
    'ADV_total',
    ['HADV_total', 'VADV_total'],
    r'$\mathrm{ADV}_{\mathrm{total}} = \mathrm{HADV}_{\mathrm{total}} + \mathrm{VADV}_{\mathrm{total}}$',
    NOTEBOOK_DIR / 'moisture_budget_decomp_regression_TARGETBOX_adv_total.png',
)
